# cuTile Python: Matrix Multiplication - SOLUTION

The vector-add notebook assigned one block to each one-dimensional output tile. Matrix multiplication extends that idea to two-dimensional output tiles and adds a reduction across the shared dimension.

Matrix multiplication is the core operation in dense neural-network layers, attention, scientific simulation, and many other GPU workloads. In this notebook, you will inspect and run `C = A @ B` with `ct.matmul`, then implement a useful fused neural-network epilogue.

In [ ]:
import os

_install_marker = os.path.expanduser(
    "~/.accelerated-computing-hub-installed"
)

# Bootstrap dependencies only in Google Colab.
if os.getenv("COLAB_RELEASE_TAG") and not os.path.exists(_install_marker):
  try:
    import cuda.tile
    import cupy
    from numba import cuda
  except ImportError:
    print("Installing PIP packages.")
    !pip install --upgrade "cuda-tile[tileiras]==1.4.0" "cuda-toolkit==13.2.1" "cuda-python==13.2.0" "cupy-cuda13x==14.2.0" "numba==0.63.1" "numba-cuda[cu13]==0.30.4" > /dev/null 2>&1
  open(_install_marker, "a").close()

In [ ]:
import cuda.tile as ct
import cupy as cp

## Tiling Matrix Multiplication

For `A` with shape `M x K` and `B` with shape `K x N`, the output `C = A @ B` has shape `M x N`. Each output element is a reduction over the shared `K` dimension:

$$C_{ij} = \sum_{k=0}^{K-1} A_{ik} B_{kj}$$

Each cuTile block computes one `tm x tn` output tile. `ct.bid(0)` selects a group of output rows and `ct.bid(1)` selects a group of output columns. The block walks across `K` in chunks of `tk`:

```
A tile: (tm x tk)  @  B tile: (tk x tn)  ->  partial C tile: (tm x tn)
```

`ct.matmul` performs the multiply and reduction for each pair of input tiles. We add those partial products to a float32 accumulator until the block has covered the full `K` dimension, then cast to the output type. Each tile-shape dimension must be a positive power of two, and the shape must be known at compile time. Zero padding makes the same kernel work when a matrix dimension is not an exact multiple of its tile size.

In [ ]:
@ct.kernel
def matmul_tile(A: ct.Array, B: ct.Array, C: ct.Array,
                tm: ct.Constant[int], tn: ct.Constant[int],
                tk: ct.Constant[int]):
  row = ct.bid(0)
  col = ct.bid(1)
  num_k_tiles = ct.num_tiles(A, axis=1, shape=(tm, tk))
  accumulator = ct.full((tm, tn), 0.0, dtype=ct.float32)

  for k in range(num_k_tiles):
    a_tile = ct.load(
      A, index=(row, k), shape=(tm, tk),
      padding_mode=ct.PaddingMode.ZERO
    )
    b_tile = ct.load(
      B, index=(k, col), shape=(tk, tn),
      padding_mode=ct.PaddingMode.ZERO
    )
    accumulator = accumulator + ct.matmul(a_tile, b_tile)

  result = ct.astype(accumulator, C.dtype)
  ct.store(C, index=(row, col), tile=result)

## Run and Validate

The dimensions below deliberately do not divide evenly by the tile sizes. The final blocks load zero-padded edge tiles, and out-of-bounds elements are discarded when the result is stored.

In [ ]:
cp.random.seed(42)
m, k, n = 130, 194, 98
tm = tn = tk = 32

A = cp.random.standard_normal((m, k), dtype=cp.float32)
B = cp.random.standard_normal((k, n), dtype=cp.float32)
C = cp.zeros((m, n), dtype=cp.float32)

grid = (ct.cdiv(m, tm), ct.cdiv(n, tn), 1)
ct.launch(cp.cuda.get_current_stream(), grid, matmul_tile,
          (A, B, C, tm, tn, tk))

cp.testing.assert_allclose(C, A @ B, rtol=1e-4, atol=2e-4)
print("Matrix multiplication OK")

## Exercise: Fused GEMM + Bias + ReLU

A dense neural-network layer commonly fuses a GEMM with bias and ReLU:

$$Y = \operatorname{ReLU}(XW + b)$$

The matrix multiplication produces a batch of output features, the one-dimensional bias `b` is broadcast across every row, and ReLU replaces negative results with zero. Running those as separate array operations writes and rereads intermediate matrices. A fused kernel applies the bias and activation while the output tile is still local.

The completed kernel reuses the tiled matrix-multiplication loop, loads the bias values for the current output columns, broadcasts them with `ct.broadcast_to`, adds them to the accumulator, applies `ct.maximum(..., 0.0)`, and stores the result.

In [ ]:
@ct.kernel
def linear_relu_tile(X: ct.Array, W: ct.Array, bias: ct.Array, Y: ct.Array,
                     tm: ct.Constant[int], tn: ct.Constant[int],
                     tk: ct.Constant[int]):
  row = ct.bid(0)
  col = ct.bid(1)
  num_k_tiles = ct.num_tiles(X, axis=1, shape=(tm, tk))
  accumulator = ct.full((tm, tn), 0.0, dtype=ct.float32)

  for k in range(num_k_tiles):
    x_tile = ct.load(
      X, index=(row, k), shape=(tm, tk),
      padding_mode=ct.PaddingMode.ZERO
    )
    w_tile = ct.load(
      W, index=(k, col), shape=(tk, tn),
      padding_mode=ct.PaddingMode.ZERO
    )
    accumulator = accumulator + ct.matmul(x_tile, w_tile)

  bias_tile = ct.load(
    bias, index=(col,), shape=(tn,),
    padding_mode=ct.PaddingMode.ZERO
  )
  bias_2d = ct.broadcast_to(bias_tile, (tm, tn))
  output = ct.maximum(accumulator + bias_2d, 0.0)
  output = ct.astype(output, Y.dtype)
  ct.store(Y, index=(row, col), tile=output)


bias = cp.random.standard_normal((n,), dtype=cp.float32)
Y = cp.zeros((m, n), dtype=cp.float32)

ct.launch(cp.cuda.get_current_stream(), grid, linear_relu_tile,
          (A, B, bias, Y, tm, tn, tk))
expected = cp.maximum(A @ B + bias, 0.0)
cp.testing.assert_allclose(Y, expected, rtol=1e-4, atol=2e-4)
print("Fused linear layer OK")

Each block owns one output tile, and the loop over `K` is the reduction that builds that tile. Once the accumulator is complete, simple epilogues such as bias, activation, scaling, or clipping can be fused before the single global-memory store.